# Embryo Diffusion — Phase‑Guided (silver F0) & Optional Classifier Guidance

This notebook trains a **conditional diffusion model** on the *silver* dataset (`data/embryo_dataset_silver/F0`) using the project's new structure. It conditions on the **morphokinetic phase** and supports **optional guidance** at sampling time using your pretrained **phase classifier** (external classifier guidance).  

**Project structure assumptions** (edit paths in the next cell if needed):
```
thesis/
├── data/
│   ├── embryo_dataset_bronze/                 # raw frames (unused here)
│   └── embryo_dataset_silver/
│       └── F0/
│           └── <embryo_id>/
│               ├── 00001.jpg, 00002.jpg, ...  # preprocessed frames (presence-filtered)
│               └── <embryo_id>_phases.csv     # copied unchanged
├── preprocessing/
│   └── embryo_presence_models/
├── classifier/
│   └── embryo_phase_models/
└── diffusion/
    ├── train_diffusion.ipynb                  # ← this notebook (recommended location)
    ├── embryo_diffusion_models/               # checkpoints
    └── diffusion_samples/                     # generated samples
```

**Notes**
- We **do not** run presence filtering here; *silver* data is already filtered by your preprocessing pipeline.
- Training uses pairs \((x_t, x_{t+\Delta})\) within each embryo sequence, with phases read from `<embryo_id>_phases.csv`.
- The UNet is **phase‑conditioned** (classifier‑free guidance during training/sampling).  
- Optionally, you can enable **external classifier guidance** at sampling using your phase classifier in `classifier/embryo_phase_models/`. If it fails to load, sampling gracefully falls back to classifier‑free guidance.


In [124]:
# ==== CONFIG (edit if needed) ====
from pathlib import Path
import os

# DEBUG MODE: Set to True for quick testing with minimal data
DEBUG_MODE = False  # Set to True to run with reduced data for testing

# If you open this notebook from thesis/diffusion/, PROJECT_ROOT points there
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'diffusion':
    # try to infer if opened from repo root
    if (PROJECT_ROOT / 'diffusion').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'diffusion'

DATA_ROOT = PROJECT_ROOT.parent / 'data'
SILVER_ROOT = DATA_ROOT / 'embryo_dataset_silver'
FOCAL_PLANES = ['F0', 'F-15', 'F-30', 'F-45', 'F15', 'F30', 'F45']
PHASE_CSV_AT_SILVER = True  # phases CSV lives next to frames in silver

# Models
PHASE_MODEL_DIR = PROJECT_ROOT.parent / 'classifier' / 'embryo_phase_models'
PHASE_MODEL_PATH = None  # autodetected if None

# Outputs
OUT_MODELS_DIR = PROJECT_ROOT / 'embryo_diffusion_models'
OUT_SAMPLES_DIR = PROJECT_ROOT / 'diffusion_samples'
CACHE_DIR = PROJECT_ROOT / 'cache'
for d in (OUT_MODELS_DIR, OUT_SAMPLES_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Training
IMG_SIZE = 256           # images are 500x500 in dataset; we center-crop & resize
BATCH_SIZE = 16
LR = 1e-4  # Reduced from 2e-4 to prevent gradient explosion
EPOCHS = 20 if not DEBUG_MODE else 2  # Reduced epochs for debug
GRAD_CLIP = 1.0  # Gradient clipping to prevent NaN
NUM_WORKERS = 8 if not DEBUG_MODE else 8  # Disable workers in debug
GRAD_ACCUM = 1
VAL_SPLIT = 0.05
MAX_SAMPLES_PER_EMBRYO = 100 if not DEBUG_MODE else 10  # Fewer samples in debug
DELTA_MIN = 1
DELTA_MAX = 5

# Diffusion
TIMESTEPS = 500  # Reduced from 1000 for 2x speedup with minimal quality loss
BETA_START, BETA_END = 1e-4, 2e-2
GUIDANCE_PROB = 0.9     # classifier-free guidance dropout during training
GUIDANCE_SCALE = 2.0    # CFG scale at sampling

# External classifier guidance (optional). If True and a classifier loads, use gradient guidance at sampling.
USE_CLASSIFIER_GUIDANCE = True
CLASSIFIER_GUIDANCE_SCALE = 1.5  # strength for external classifier guidance step
CLASSIFIER_INPUT_SIZE = 256      # if your classifier expects another size, change this

print('Project root:', PROJECT_ROOT)
print('Silver root:', SILVER_ROOT)
print('Focal planes:', FOCAL_PLANES)
if DEBUG_MODE:
    print('⚠️  DEBUG_MODE is ON - using reduced data for testing')
for plane in FOCAL_PLANES:
    plane_dir = SILVER_ROOT / plane
    assert plane_dir.exists(), f"Silver {plane} folder not found: {plane_dir}"



Project root: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion
Silver root: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/data/embryo_dataset_silver
Focal planes: ['F0', 'F-15', 'F-30', 'F-45', 'F15', 'F30', 'F45']


In [125]:
# Imports & utils
import math, random, glob, re, os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Diffusers library for UNet and schedulers
try:
    from diffusers import UNet2DModel, DDPMScheduler
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'diffusers'])
    from diffusers import UNet2DModel, DDPMScheduler

In [126]:
# Initialize seed and device (must be before using PHASE_LABELS in later cells)
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PHASE_LABELS = ['pPB2','pPNa','pPNf','p2','p3','p4','p5','p6','p7','p8','p9+','pM','pSB','pB','pEB','pHB']
NUM_PHASES = len(PHASE_LABELS)

# Plane ID mapping
PLANE_TO_ID = {plane: i for i, plane in enumerate(FOCAL_PLANES)}
ID_TO_PLANE = {i: plane for plane, i in PLANE_TO_ID.items()}
NUM_PLANES = len(FOCAL_PLANES)


Using device: cuda


In [127]:
# Build frame index from all focal planes: phase CSV lives next to frames
INDEX_CACHE = CACHE_DIR / 'silver_all_planes_frame_index.csv'

def build_index_from_silver():
    rows = []
    for plane in FOCAL_PLANES:
        plane_root = SILVER_ROOT / plane
        embryo_ids = sorted([d.name for d in plane_root.iterdir() if d.is_dir()])
        for eid in tqdm(embryo_ids, desc=f'Indexing {plane}'):
            folder = plane_root / eid
            ann_path = folder / f"{eid}_phases.csv"
            if not ann_path.exists():
                continue
            # Use .jpeg files
            img_files = sorted(glob.glob(str(folder / '*.jpeg')))
            if not img_files:
                continue
            # Extract frame numbers from filenames (00037.jpeg → 37)
            frame_numbers = {}
            for f in img_files:
                basename = os.path.basename(f)
                m = re.search(r'(\d+)\.jpeg$', basename)
                if m:
                    frame_num = int(m.group(1))
                    frame_numbers[frame_num] = f
            # read CSV (phase, start, end) with 1-based frame indices
            df = pd.read_csv(ann_path, header=None, names=['phase','start','end'], sep='[,;\s]+', engine='python')
            def phase_to_id(ph: str):
                ph = str(ph).strip()
                ph = ph.replace('t','p',1) if ph.startswith('t') else ph  # accept t* or p*
                return PHASE_LABELS.index(ph)
            label_spans = []
            for _, r in df.iterrows():
                try:
                    pid = phase_to_id(r['phase'])
                    label_spans.append((int(r['start']), int(r['end']), pid))
                except Exception:
                    pass
            # assign phase per frame using actual frame numbers from filenames
            for frame_num, path in frame_numbers.items():
                pid = None
                for s, e, p in label_spans:
                    if s <= frame_num <= e:  # frame_num is 1-based, matches CSV
                        pid = p
                        break
                if pid is None:
                    continue
                rows.append({'embryo_id': eid, 'plane': plane, 'frame_num': frame_num, 'path': path, 'phase_id': pid})
    return pd.DataFrame(rows)

if INDEX_CACHE.exists():
    index_df = pd.read_csv(INDEX_CACHE)
else:
    index_df = build_index_from_silver()
    index_df.to_csv(INDEX_CACHE, index=False)

print('Indexed frames:', len(index_df))
print('Planes:', index_df['plane'].unique().tolist())
index_df.head()



Indexed frames: 2028328
Planes: ['F0', 'F-15', 'F-30', 'F-45', 'F15', 'F30', 'F45']


,embryo_id,plane,frame_num,path,phase_id
0,AA83-7,F0,5,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
1,AA83-7,F0,6,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
2,AA83-7,F0,7,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
3,AA83-7,F0,8,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
4,AA83-7,F0,9,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0


In [128]:
# Build training pairs (x_t -> x_{t+Δ}) within each (embryo, plane), only using indexed frames
# Pairs from same plane only
PAIRS_CACHE = CACHE_DIR / f'pairs_silver_all_planes_d{DELTA_MIN}-{DELTA_MAX}_max{MAX_SAMPLES_PER_EMBRYO}.csv'

def build_pairs(df: pd.DataFrame, delta_min=DELTA_MIN, delta_max=DELTA_MAX, max_samples_per_embryo=MAX_SAMPLES_PER_EMBRYO):
    pairs = []
    by_embryo_plane = df.groupby(['embryo_id', 'plane'])
    for (eid, plane), g in tqdm(by_embryo_plane, desc='Sampling pairs per (embryo, plane)'):
        g = g.sort_values('frame_num')
        files = g['path'].tolist()
        phases = g['phase_id'].tolist()
        frame_nums = g['frame_num'].tolist()  # for delta_t calculation
        n = len(files)
        if n < delta_min + 1:
            continue
        cnt = 0
        tries = 0
        while cnt < max_samples_per_embryo and tries < max(1000, 5*max_samples_per_embryo):
            tries += 1
            i = random.randrange(0, n-1)
            d = random.randint(delta_min, min(delta_max, n-1-i))
            j = i + d
            delta_t = frame_nums[j] - frame_nums[i]  # actual temporal difference
            pairs.append({'embryo_id': eid, 'plane': plane,
                          'src_path': files[i], 'tgt_path': files[j],
                          'src_phase': int(phases[i]), 'tgt_phase': int(phases[j]),
                          'delta_t': int(delta_t)})
            cnt += 1
    return pd.DataFrame(pairs)

if INDEX_CACHE.exists():
    index_df = pd.read_csv(INDEX_CACHE)
else:
    index_df = build_index_from_silver()
    index_df.to_csv(INDEX_CACHE, index=False)

# In DEBUG mode, sample only a few embryos
if DEBUG_MODE:
    unique_embryos = index_df['embryo_id'].unique()
    if len(unique_embryos) > 3:
        debug_embryos = np.random.choice(unique_embryos, 3, replace=False)
        index_df = index_df[index_df['embryo_id'].isin(debug_embryos)]
        print(f'DEBUG_MODE: Using only {len(debug_embryos)} embryos for testing')

print('Indexed frames:', len(index_df))
print('Planes:', index_df['plane'].unique().tolist())

# Build pairs from indexed frames
if PAIRS_CACHE.exists():
    pairs_df = pd.read_csv(PAIRS_CACHE)
else:
    pairs_df = build_pairs(index_df)
    pairs_df.to_csv(PAIRS_CACHE, index=False)

print('Total training pairs:', len(pairs_df))
print('Pairs per plane:', pairs_df['plane'].value_counts().sort_index())

# Split by embryo_id (plane-balanced: same embryos in train/val across all planes)
embryos = pairs_df['embryo_id'].unique().tolist()
random.shuffle(embryos)
val_count = max(1, int(len(embryos) * VAL_SPLIT))
val_embryos = set(embryos[:val_count])
train_df = pairs_df[~pairs_df['embryo_id'].isin(val_embryos)].reset_index(drop=True)
val_df   = pairs_df[pairs_df['embryo_id'].isin(val_embryos)].reset_index(drop=True)
print(f'Embryos total={len(embryos)}, train={len(embryos)-val_count}, val={val_count}')
print('Train pairs:', len(train_df), 'Val pairs:', len(val_df))
print('Train pairs per plane:', train_df['plane'].value_counts().sort_index())
print('Val pairs per plane:', val_df['plane'].value_counts().sort_index())


Indexed frames: 2028328
Planes: ['F0', 'F-15', 'F-30', 'F-45', 'F15', 'F30', 'F45']
Total training pairs: 492100
Pairs per plane: plane
F-15    70300
F-30    70300
F-45    70300
F0      70300
F15     70300
F30     70300
F45     70300
Name: count, dtype: int64
Embryos total=703, train=668, val=35
Train pairs: 467600 Val pairs: 24500
Train pairs per plane: plane
F-15    66800
F-30    66800
F-45    66800
F0      66800
F15     66800
F30     66800
F45     66800
Name: count, dtype: int64
Val pairs per plane: plane
F-15    3500
F-30    3500
F-45    3500
F0      3500
F15     3500
F30     3500
F45     3500
Name: count, dtype: int64


In [129]:
# Diffusion schedule using diffusers
from diffusers import DDPMPipeline

scheduler = DDPMScheduler(
    num_train_timesteps=TIMESTEPS,
    beta_start=BETA_START,
    beta_end=BETA_END,
    beta_schedule='scaled_linear',
    variance_type='fixed_small'
)

# Pre-compute noise schedule for training
betas = scheduler.betas.to(device)
alphas = (1 - betas).to(device)
alphas_cumprod = torch.cumprod(alphas, dim=0).to(device)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# Compute posterior variance for DDPM sampling
# posterior_var_t = (1 - alpha_cumprod_{t-1}) / (1 - alpha_cumprod_t) * beta_t
alphas_cumprod_prev = torch.cat([torch.ones(1, device=device), alphas_cumprod[:-1]])
posterior_var = (1 - alphas_cumprod_prev) / (1 - alphas_cumprod) * betas
posterior_var = torch.clamp(posterior_var, min=1e-20)  # avoid log(0)


In [130]:
# Simplified diffusion model using diffusers UNet2DModel

class ConditioningModule(nn.Module):
    """Embeds phase, plane, and delta_t into a single conditioning vector."""
    def __init__(self, num_classes=NUM_PHASES, num_planes=NUM_PLANES, max_delta_t=DELTA_MAX, cond_dim=128):
        super().__init__()
        emb_dim = cond_dim // 3
        remainder = cond_dim % 3
        class_emb_dim = emb_dim + remainder
        self.class_emb = nn.Embedding(num_classes + 1, class_emb_dim)  # +1 for null token (CFG)
        self.plane_emb = nn.Embedding(num_planes, emb_dim)
        self.delta_t_emb = nn.Embedding(max_delta_t, emb_dim)
        self.null_class = num_classes
        self.cond_dim = cond_dim
    
    def forward(self, cls, plane, delta_t):
        cls_emb = self.class_emb(cls.clamp(0, self.class_emb.num_embeddings - 1))
        plane_emb = self.plane_emb(plane.clamp(0, self.plane_emb.num_embeddings - 1))
        delta_t_emb = self.delta_t_emb(delta_t.clamp(0, self.delta_t_emb.num_embeddings - 1))
        c_emb = torch.cat([cls_emb, plane_emb, delta_t_emb], dim=1)
        return c_emb

class DiffusionCond(nn.Module):
    """Lightweight diffusion model using diffusers UNet2DModel."""
    def __init__(self, cond_dim=128):
        super().__init__()
        # UNet2DModel from diffusers - uses only simple blocks (no attention)
        self.unet = UNet2DModel(
            sample_size=IMG_SIZE,
            in_channels=2,
            out_channels=1,
            layers_per_block=2,
            block_out_channels=(32, 64, 128, 128),
            down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D"),
            up_block_types=("UpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
        )
        self.conditioning = ConditioningModule(cond_dim=cond_dim)
    
    def forward(self, x_concat, t, cls, plane, delta_t):
        # x_concat: [B, 2, H, W], t: [B], cls/plane/delta_t: [B]
        # UNet2DModel expects timesteps as a tensor
        noise = self.unet(x_concat, t).sample  # [B, 1, H, W]
        return noise
    
    def q_sample(self, x0, t, noise=None):
        """Add noise to x0."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha = sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_alpha * x0 + sqrt_one_minus * noise, noise
    
    def p_losses(self, x_tgt, x_src, cond_cls, plane_id, delta_t):
        """Training loss."""
        B = x_tgt.size(0)
        t = torch.randint(0, TIMESTEPS, (B,), device=x_tgt.device, dtype=torch.long)
        x_noisy, noise = self.q_sample(x_tgt, t)
        x_concat = torch.cat([x_noisy, x_src], dim=1)
        
        # Classifier-free guidance dropout (drop conditioning during training)
        use_cond = (torch.rand(B, device=x_tgt.device) < GUIDANCE_PROB).long()
        cls_in = cond_cls.clone()
        cls_in[use_cond == 0] = self.conditioning.null_class
        
        pred = self.forward(x_concat, t, cls_in, plane_id, delta_t)
        return F.mse_loss(pred, noise)

In [131]:
def load_image(path: str, size: int = IMG_SIZE):
    img = Image.open(path).convert('L')
    # center-crop square then resize
    w, h = img.size
    m = min(w, h)
    left = (w - m) // 2
    top = (h - m) // 2
    img = img.crop((left, top, left + m, top + m))
    if size is not None and (img.size[0] != size or img.size[1] != size):
        img = img.resize((size, size), Image.BICUBIC)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = arr * 2 - 1  # [-1, 1]
    return torch.from_numpy(arr).unsqueeze(0)

def denorm(x):
    return (x.clamp(-1,1) + 1) / 2

# Dataset class
class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_src = load_image(row['src_path'])
        x_tgt = load_image(row['tgt_path'])
        src_phase = int(row['src_phase'])
        tgt_phase = int(row['tgt_phase'])
        plane_id = PLANE_TO_ID[row['plane']]
        delta_t = int(row['delta_t']) - 1  # 0-indexed for embedding
        return x_src, x_tgt, src_phase, tgt_phase, plane_id, delta_t

# Create data loaders
train_dataset = PairDataset(train_df)
val_dataset = PairDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)



In [132]:
diffusion = DiffusionCond(cond_dim=128).to(device)
optimizer = torch.optim.AdamW(diffusion.unet.parameters(), lr=LR, betas=(0.9, 0.999))

# Setup DDPMPipeline for sampling (initialized later after training)
@torch.no_grad()
def sample_from_model(shape, cls, plane, delta_t, guidance_scale=GUIDANCE_SCALE, x_src=None, 
                     classifier=None, classifier_scale=CLASSIFIER_GUIDANCE_SCALE):
    """Generate samples using the trained model."""
    B = shape[0]
    x_noisy = torch.randn((B, 1, shape[2], shape[3]), device=device)
    if x_src is None:
        x_src = torch.zeros_like(x_noisy)
    x_concat = torch.cat([x_noisy, x_src], dim=1)
    
    # Denoising loop
    for t_idx in reversed(range(TIMESTEPS)):
        t_tensor = torch.full((B,), t_idx, device=device, dtype=torch.long)
        
        # Predict noise with conditioning
        eps_cond = diffusion.forward(x_concat, t_tensor, cls, plane, delta_t)
        
        # Predict noise without conditioning (CFG)
        null_cls = torch.full_like(cls, diffusion.conditioning.null_class)
        eps_null = diffusion.forward(x_concat, t_tensor, null_cls, plane, delta_t)
        
        # Apply classifier-free guidance
        eps = eps_null + guidance_scale * (eps_cond - eps_null)
        
        # Denoise step
        beta_t = scheduler.betas[t_idx]
        alpha_t = scheduler.alphas[t_idx]
        sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t_idx]
        
        mean = (x_concat[:, :1] - beta_t / sqrt_one_minus * eps) / torch.sqrt(alpha_t)
        var = posterior_var[t_idx] if t_idx > 0 else torch.tensor(0.0, device=device)
        noise = torch.randn_like(x_concat[:, :1]) if t_idx > 0 else torch.zeros_like(x_concat[:, :1])
        x_new = mean + torch.sqrt(var) * noise
        
        x_concat = torch.cat([x_new, x_concat[:, 1:2]], dim=1)
        
        # Optional: classifier guidance
        if classifier is not None and t_idx % 10 == 0:  # Apply every 10 steps to save compute
            x_noisy_part = x_concat[:, :1].detach()
            x_noisy_part.requires_grad_(True)
            x_for_clf = (x_noisy_part + 1) / 2
            logits = classifier(x_for_clf)
            logp = F.log_softmax(logits, dim=1)[torch.arange(B, device=device), cls]
            loss = -logp.mean()
            g = torch.autograd.grad(loss, x_for_clf, retain_graph=False)[0]
            x_for_clf = (x_for_clf - classifier_scale * g).detach()
            x_noisy_part = (x_for_clf * 2 - 1).detach()
            x_concat = torch.cat([x_noisy_part, x_concat[:, 1:2]], dim=1)
    
    return x_concat[:, :1]


In [133]:
# Try to auto-detect a phase classifier checkpoint
def autodetect_model(model_dir: Path, keywords=('phase','classifier','model','.pt','.pth','.ckpt','.jit')):
    if not model_dir.exists():
        return None
    cands = []
    for p in model_dir.glob('**/*'):
        name = p.name.lower()
        if p.is_file() and any(k in name for k in keywords):
            cands.append(p)
    if cands:
        cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        return cands[0]
    return None

if PHASE_MODEL_PATH is None:
    PHASE_MODEL_PATH = autodetect_model(PHASE_MODEL_DIR)
print('Phase classifier candidate:', PHASE_MODEL_PATH)

# We don't know your exact classifier class; we attempt a few robust load paths.
phase_classifier = None
phase_classifier_input_resize = CLASSIFIER_INPUT_SIZE

def _resize_for_classifier(x):
    # x: [B,1,H,W] in [-1,1] → resize to CLASSIFIER_INPUT_SIZE
    if x.shape[-1] != phase_classifier_input_resize:
        x = F.interpolate(x, size=(phase_classifier_input_resize, phase_classifier_input_resize), mode='bilinear', align_corners=False)
    return x

if PHASE_MODEL_PATH is not None and USE_CLASSIFIER_GUIDANCE:
    try:
        # 1) TorchScript
        phase_classifier = torch.jit.load(str(PHASE_MODEL_PATH), map_location=device)
        phase_classifier.eval()
    except Exception:
        try:
            # 2) Full nn.Module saved in checkpoint
            obj = torch.load(PHASE_MODEL_PATH, map_location=device)
            if isinstance(obj, nn.Module):
                phase_classifier = obj
                phase_classifier.eval()
            elif isinstance(obj, dict) and 'model' in obj and isinstance(obj['model'], nn.Module):
                phase_classifier = obj['model']
                phase_classifier.eval()
            else:
                phase_classifier = None
        except Exception:
            phase_classifier = None

if phase_classifier is not None:
    # Wrapper to accept [-1,1] grayscale from diffusion and convert to [0,1] for classifier
    # The phase classifier was trained with Normalize([0.5], [0.5]) applied to [0,1] inputs.
    # The model expects raw [0,1] inputs, then applies the normalization internally:
    # Normalize([0.5], [0.5]) converts [0,1] → [-1,1] via (x - 0.5) / 0.5 = 2*x - 1
    class PhaseClassifierWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            # x: [B, 1, H, W] in [-1, 1] (from diffusion model)
            x = _resize_for_classifier(x)
            # Convert to [0, 1] (what classifier expects before its Normalize)
            x_01 = (x + 1) / 2  # [-1, 1] → [0, 1]
            # if the classifier expects 3ch, replicate
            if getattr(self.model, 'in_channels', 1) == 3 or any(getattr(m, 'in_channels', 1)==3 for m in self.model.modules() if hasattr(m,'in_channels')):
                x_01 = x_01.repeat(1, 3, 1, 1)
            # Classifier applies Normalize([0.5], [0.5]) internally: [0,1] → [-1,1]
            return self.model(x_01)
    phase_classifier = PhaseClassifierWrapper(phase_classifier).to(device)
    print('Phase classifier loaded for external guidance.')
else:
    print('Phase classifier not available or failed to load. External guidance will be disabled.')


def save_tensor_image(tensor, path):
    """Save tensor as grayscale image."""
    arr = (denorm(tensor).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    Image.fromarray(arr).save(path)


Phase classifier candidate: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/classifier/embryo_phase_models/phase_classifier_all_focal_planes_f1=643.pth


/tmp/ipykernel_106527/1159883401.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(PHASE_MODEL_PATH, map_location=device)


Phase classifier loaded for external guidance.


In [134]:
# Train loop - use new diffusers-based diffusion model
best_val = None
ckpt_path = OUT_MODELS_DIR / f'phase_plane_guided_ddpm_{IMG_SIZE}px.pt'

# Early stopping config
EARLY_STOPPING_PATIENCE = 10  # set to None or 0 to disable
epochs_no_improve = 0
SAMPLE_EVERY_N_EPOCHS = 1  # Generate samples after every N epochs

# GradScaler for proper mixed precision training (prevents NaN from underflow/overflow)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

def one_epoch(loader, train=True):
    diffusion.train(mode=train)
    total = 0.0
    count = 0
    nan_batches = 0
    for x_src, x_tgt, src_phase, tgt_phase, plane_id, delta_t in tqdm(loader):
        x_src = x_src.to(device, non_blocking=True)
        x_tgt = x_tgt.to(device, non_blocking=True)
        plane_id = plane_id.to(device, non_blocking=True)
        delta_t = delta_t.to(device, non_blocking=True)
        cond_cls = torch.as_tensor(tgt_phase, dtype=torch.long, device=device)
        
        # Mixed precision training with proper GradScaler
        with torch.autocast(device_type='cuda', enabled=use_amp):
            loss = diffusion.p_losses(x_tgt, x_src, cond_cls, plane_id, delta_t)
        
        # Check for NaN/Inf BEFORE backward (critical: don't corrupt model with bad gradients)
        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            if nan_batches <= 3:  # Only warn first few times
                print(f"WARNING: NaN/Inf loss detected, skipping batch (total: {nan_batches})")
            continue
        
        if train:
            optimizer.zero_grad(set_to_none=True)
            # Use scaler for backward pass
            scaler.scale(loss).backward()
            # Unscale before clipping
            scaler.unscale_(optimizer)
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(diffusion.parameters(), GRAD_CLIP)
            # Step with scaler (skips step if gradients contain inf/nan)
            scaler.step(optimizer)
            scaler.update()
        
        total += loss.item() * x_tgt.size(0)
        count += x_tgt.size(0)
    
    if nan_batches > 0:
        print(f"  (Epoch had {nan_batches} NaN/Inf batches skipped)")
    return total / max(1, count)

for epoch in range(1, EPOCHS+1):
    tl = one_epoch(train_loader, train=True)
    with torch.no_grad():
        vl = one_epoch(val_loader, train=False)
    print(f"[Epoch {epoch:03d}] train {tl:.4f} | val {vl:.4f}")
    if best_val is None or vl < best_val - 1e-8:
        best_val = vl
        epochs_no_improve = 0
        torch.save({'model': diffusion.state_dict(), 'epoch': epoch, 'val_loss': vl, 'config': {
            'IMG_SIZE': IMG_SIZE, 'TIMESTEPS': TIMESTEPS, 'PHASE_LABELS': PHASE_LABELS,
            'FOCAL_PLANES': FOCAL_PLANES, 'NUM_PLANES': NUM_PLANES
        }}, ckpt_path)
        print('Saved best to', ckpt_path)
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")
    
    # Generate samples after every N epochs
    if epoch % SAMPLE_EVERY_N_EPOCHS == 0:
        print(f"\n--- Generating samples for epoch {epoch} ---")
        diffusion.eval()
        with torch.no_grad():
            for plane in ['F0', 'F-15', 'F30']:  # Sample from a few planes
                sample_dir = OUT_SAMPLES_DIR / f'epoch_{epoch:03d}' / plane
                sample_dir.mkdir(parents=True, exist_ok=True)
                # Sample 3 random validation pairs
                plane_data = val_df[val_df['plane'] == plane]
                if len(plane_data) > 0:
                    samp = plane_data.sample(n=min(3, len(plane_data)))
                    for idx, (_, row) in enumerate(samp.iterrows()):
                        x_src = load_image(row['src_path']).to(device)[None]
                        tgt_cls = torch.tensor([int(row['tgt_phase'])], device=device, dtype=torch.long)
                        plane_id = torch.tensor([PLANE_TO_ID[plane]], device=device, dtype=torch.long)
                        delta_t_tensor = torch.tensor([int(row['delta_t']) - 1], device=device, dtype=torch.long)
                        x_gen = sample_from_model((1, 1, IMG_SIZE, IMG_SIZE), tgt_cls, plane_id, delta_t_tensor,
                                                 guidance_scale=GUIDANCE_SCALE, x_src=x_src, classifier=None)
                        save_path = sample_dir / f'sample_{idx:02d}.jpg'
                        save_tensor_image(x_gen[0], save_path)
        print(f"Samples saved to {OUT_SAMPLES_DIR / f'epoch_{epoch:03d}'}\n")
    
    # Early stopping check
    if EARLY_STOPPING_PATIENCE and EARLY_STOPPING_PATIENCE > 0 and epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f"Stopping early after {epochs_no_improve} epochs without improvement (patience={EARLY_STOPPING_PATIENCE}).")
        break

print(f"\nTraining complete! Best checkpoint saved to {ckpt_path}")



100%|██████████| 1532/1532 [02:55<00:00,  8.72it/s]


[Epoch 001] train 0.0352 | val 0.0314
Saved best to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/embryo_diffusion_models/phase_plane_guided_ddpm_256px.pt

--- Generating samples for epoch 1 ---
Samples saved to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/diffusion_samples/epoch_001



100%|██████████| 1532/1532 [02:56<00:00,  8.69it/s]


[Epoch 002] train 0.0310 | val 0.0303
Saved best to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/embryo_diffusion_models/phase_plane_guided_ddpm_256px.pt

--- Generating samples for epoch 2 ---
Samples saved to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/diffusion_samples/epoch_002



100%|██████████| 1532/1532 [02:52<00:00,  8.89it/s]


[Epoch 003] train 0.0299 | val 0.0298
Saved best to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/embryo_diffusion_models/phase_plane_guided_ddpm_256px.pt

--- Generating samples for epoch 3 ---
Samples saved to /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion/diffusion_samples/epoch_003



 12%|█▏        | 3502/29225 [21:47<2:40:01,  2.68it/s]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def visualize_samples(num=4, plane='F0', use_classifier_guidance: bool = True):
    """Visualize random samples from validation set."""
    plane_id = PLANE_TO_ID[plane]
    samp = val_df[val_df['plane'] == plane].sample(n=min(num, len(val_df[val_df['plane'] == plane]))).reset_index(drop=True)
    for i in range(len(samp)):
        row = samp.iloc[i]
        x_src = load_image(row['src_path']).to(device)[None]
        tgt_cls = torch.tensor([int(row['tgt_phase'])], device=device, dtype=torch.long)
        plane_tensor = torch.tensor([plane_id], device=device, dtype=torch.long)
        delta_t_tensor = torch.tensor([int(row['delta_t']) - 1], device=device, dtype=torch.long)
        clf = phase_classifier if (use_classifier_guidance and phase_classifier is not None and USE_CLASSIFIER_GUIDANCE) else None
        
        if clf is not None:
            with torch.enable_grad():
                x_gen = sample_from_model((1, 1, IMG_SIZE, IMG_SIZE), tgt_cls, plane_tensor, delta_t_tensor, 
                                         guidance_scale=GUIDANCE_SCALE, x_src=x_src, classifier=clf)
        else:
            x_gen = sample_from_model((1, 1, IMG_SIZE, IMG_SIZE), tgt_cls, plane_tensor, delta_t_tensor,
                                     guidance_scale=GUIDANCE_SCALE, x_src=x_src, classifier=None)
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 3, 1)
        plt.title(f'Source (x_t, {plane})')
        plt.imshow(denorm(x_src[0]).squeeze().cpu(), cmap='gray')
        plt.axis('off')
        plt.subplot(1, 3, 2)
        plt.title(f'Predicted (x_{{t+Δ}})')
        plt.imshow(denorm(x_gen[0]).squeeze().cpu(), cmap='gray')
        plt.axis('off')
        plt.subplot(1, 3, 3)
        plt.title('Difference')
        plt.imshow((denorm(x_gen[0]) - denorm(x_src[0])).squeeze().cpu().abs(), cmap='hot')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

@torch.no_grad()
def predict_next_frames(target_phase: str, plane: str, out_path: str, guidance_scale: float = GUIDANCE_SCALE, 
                       use_classifier_guidance: bool = True, delta_t: int = 2):
    """Generate a frame for a target phase."""
    assert target_phase in PHASE_LABELS, f"Unknown phase {target_phase}. Choices: {PHASE_LABELS}"
    assert plane in FOCAL_PLANES, f"Unknown plane {plane}. Choices: {FOCAL_PLANES}"
    cls = torch.tensor([PHASE_LABELS.index(target_phase)], device=device, dtype=torch.long)
    plane_id = torch.tensor([PLANE_TO_ID[plane]], device=device, dtype=torch.long)
    delta_t_tensor = torch.tensor([delta_t - 1], device=device, dtype=torch.long)
    clf = phase_classifier if (use_classifier_guidance and phase_classifier is not None and USE_CLASSIFIER_GUIDANCE) else None
    
    if clf is not None:
        with torch.enable_grad():
            x_gen = sample_from_model((1, 1, IMG_SIZE, IMG_SIZE), cls, plane_id, delta_t_tensor, 
                                     guidance_scale=guidance_scale, x_src=None, classifier=clf)
    else:
        x_gen = sample_from_model((1, 1, IMG_SIZE, IMG_SIZE), cls, plane_id, delta_t_tensor,
                                 guidance_scale=guidance_scale, x_src=None, classifier=None)
    
    out_path = str(out_path)
    save_tensor_image(x_gen[0], out_path)
    return out_path